# Self-RAG

Let the model reflect: is retrieval needed, is each passage relevant, is the answer supported, is it useful?

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

The original Self-RAG fine-tunes special reflection tokens. Here we reproduce the *behaviour* with prompts: `Retrieve?`, `IsRel`, `IsSup`, `IsUse`.

In [ ]:
def need_retrieval(q):  return llm_yes(f"Would looking up documents help answer this question accurately?\n\nQuestion: {q}")
def is_rel(q, p):       return llm_yes(f"Question: {q}\n\nPassage: {p}\n\nIs the passage relevant to the question?")
def is_supported(a, p): return llm_yes(f"Passage: {p}\n\nAnswer: {a}\n\nIs the answer fully supported by the passage?")
def usefulness(q, a):
    out = llm(f"Rate how useful the answer is for the question from 1 to 5. Reply with the number only.\n\nQuestion: {q}\n\nAnswer: {a}")
    m = re.search(r"[1-5]", out)
    return int(m.group()) if m else 1

Generate one candidate answer per relevant passage and keep the best-scoring one.

In [ ]:
def self_rag(query, k=4):
    if not need_retrieval(query):
        return "no retrieval", llm(query)
    candidates = []
    for h in store.search(query, k):
        if not is_rel(query, h["text"]):
            continue
        a = answer(query, [h["text"]])
        score = 2 * is_supported(a, h["text"]) + usefulness(query, a)   # supported answers win ties
        candidates.append((score, a, h["text"]))
    if not candidates:
        return "no relevant passage", llm(query)
    best = max(candidates, key=lambda c: c[0])
    return f"best of {len(candidates)} candidates (score {best[0]})", best[1]

for q in ["What does a reranker do?", "Write a haiku about autumn."]:
    how, response = self_rag(q)
    print(f"Q: {q}\n  [{how}]\n  {response[:300]}\n")